In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from hex_malware_detection import DEFAULT_DATASET, LABEL_COLUMN, load_hex_dataset, train_hex_models

In [ ]:
data = load_hex_dataset(DEFAULT_DATASET)
data.head()

In [ ]:
data.isnull().sum()

In [ ]:
sns.countplot(x='Label', data=data);

In [ ]:
X = data.drop(columns=[LABEL_COLUMN])
y = data[LABEL_COLUMN].to_numpy()

In [ ]:
model_bundle, X_test, y_test = train_hex_models(data)
metrics = model_bundle["metrics"]

In [ ]:
for name, results in metrics.items():
    print(f"{name} accuracy: {results['accuracy']:.4f}")
    display(pd.DataFrame(results["classification_report"]).T)

    confusion = results["confusion_matrix"]
    sns.heatmap(
        confusion,
        annot=True,
        fmt="d",
        xticklabels=model_bundle["classes"],
        yticklabels=model_bundle["classes"],
    )
    plt.xlabel("Predicted label")
    plt.ylabel("True label")
    plt.title(f"{name} confusion matrix")
    plt.show()

### Regularized Models: Decision Tree and K-Nearest Neighbors
The shared training workflow uses a stratified 80/20 train/test split. The decision tree is constrained by depth and minimum sample sizes; KNN uses nine neighbors with standardization fitted only on training data.

In [ ]:
accuracy_summary = pd.DataFrame(
    {
        name: {"accuracy": results["accuracy"]}
        for name, results in metrics.items()
    }
).T
accuracy_summary

### Testing the Models on a New Sample
Let's extract a sample from the test dataset and use our regularized, trained models to make predictions.

In [ ]:
sample_index = 0
test_sample = X_test.iloc[[sample_index]]
true_label = y_test.iloc[sample_index]

print(f"True (actual) label: {true_label}")
for name, model in model_bundle["models"].items():
    prediction = model.predict(test_sample)[0]
    print(f"{name} prediction: {prediction}")